# 1.0 Introduction
blah blah

## 1.1 Get all the necessary Libraries

In [1]:
import pandas as pd
import numpy as np

In [2]:
# System and OS related tasks
import sys
import os
import importlib
# Add the project root to Python path
project_root = os.path.abspath('..')
sys.path.insert(0, project_root)

# path to directories
raw_dir = '../data/raw'
processed_dir = '../data/processed'

# 2.0 Data Source: FAO Data & ETL Process

## 2.1 FAO Source Data

FAO data is downloaded from [Food and Agriculture Organization of the United Nations](https://www.fao.org/faostat/en/#data/QCL)

It is deemed to be more expeditious to use the FAO Bulk Download option for their normalised data (1 row per country-year). The csv file has the following columns:


| Area Code | Area Code (M49) | Area | Item Code | Item Code (CPC) | Item | Element Code | Element | Year Code | Year | Unit | Value | Flag | Note |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|   |   |   |   |   |   |   |   |   |   |   |   |   |   |
|   |   |   |   |   |   |   |   |   |   |   |   |   |   |


We will subset the data based on 
* year (numeric) for 2000 to 2024 inclusive
* Item code (text field)
* Area (i.e. Country)


## 2.2 Understanding Key FAO Data Columns

#### 2.2.1 By Year

According to the FAO website, their data was updated in December 2025. As they sometimes provide an estimate of the data if they are not available, we will assume that the period of analysis to be from year 2000 to 2024 inclusive.

In the second half of the ETL, we might further restrict the number of years depending on the availablity of the World's BAnk Climate and Economics data.

#### 2.2.2 By Item Code

The items that this project is interested in are:
* "Crops, Primary > (List) > Tea Leaves" (**Item Code 677**)
  * This is the raw tea leaves harvested as opposed FAO's other tea category of "Crops, Processed > (List) > Green tea (not fermented), black (fermented) and partly fermented tea, in immediate packings of a content not exceeding 3kg" (Item Code 675)
  * Setting aside the myraid types of tea processed ready for consumption, having the data in denomiations of "packings not exceeding 3kg" might complicate matters when it comes to comparing yield (with coffee)". 
  * Also, the cateogories of tea used by FAO looked at the fermentation levels. This is infact a misnormer.
    * The green teas, black teas, white teas, oolong teas and yellow teas are predominantly oxidised.
    * Fermented teas (the red teas) undergo a process different to oxisation. 
      * True fermented teas undergo a different chemical reaction then its oxidised counterparts. The fermentation occurs either 
        * in a process called "damp piling" whereby the tea leaves undergo both the oxidation and fermentation or 
        * vintaged slowly over time via the same chemical reaction much like how wine is vintaged.  
      * Although fermented tea can be sold in the year of harvest and processing, they are usually not drank in the year of harvest. 
      * Depending on the locality and expertise in vintaging such tea, its yield and value can vary widely and does not fit into the analysis proposed by this project.
      * Both of these factors add a level of complexity in its granularity when it comes to tea volume data provided by FAO.
    * Hence, this project will only use "Crops, Primary > List > Tea Leaves" as the source of data.
* "Crops, primary > (List) > Coffee, green" (**Item Code 656**)
  * There is only 1 item code of coffee listed.
  * Choosing the data granularity at the unprocessed level allows like for like comparison with the tea data.
  * ‼️‼️📝 Note: Please see in the analysis of crop dominance the need to change the granularity of the coffee and tea data.

#### 2.2.3 By Area (Country)

We might only concentrate on 10-15 countries as part of the MVP for this project.
The country name will be converted to ISO3 country name convention using the handy `country_converter` package.

## 2.3 Extraction & Load FAO Data

‼️‼️📝 Note: The original FAO bulk data Production_Crops_Livestock_E_All_Data_Normalized.csv is too large for github and will not be place in the git repo.

In [3]:
# Read in the whole FAO Bulk Data file
raw_fao_dir = '../data/raw/Production_Crops_Livestock_E_All_Data_Normalized/'
filename = "Production_Crops_Livestock_E_All_Data_Normalized.csv"

data_columns = ['Area_Code', 'Area_Code_M49', 'Area', 'Item_Code', 'Item_Code_CPC', 'Item', 'Element_Code', 'Element', 'Year_Code', 'Year', 'Unit', 'Value', 'Flag', 'Note']
column_dtypes = {
    'Area_Code': int, 'Area_Code_M49': str, 'Area' : str, 'Item_Code' : int, 'Item_Code_CPC': str, 'Item' : str, 'Element_Code' : int, 'Element' : str, 'Year_Code' : int, 'Year': int, 'Unit' : str, 'Value' : float, 'Flag' : str, 'Note' : str
}

df_fao_raw = pd.read_csv(
    f"{raw_fao_dir}/{filename}",
    header=0,
    names=data_columns,
    dtype=column_dtypes
    )
df_fao_raw.head(3)

,Area_Code,Area_Code_M49,Area,Item_Code,Item_Code_CPC,Item,Element_Code,Element,Year_Code,Year,Unit,Value,Flag,Note
0,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5312,Area harvested,1961,1961,ha,0.0,A,NaN
1,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5312,Area harvested,1962,1962,ha,0.0,A,NaN
2,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5312,Area harvested,1963,1963,ha,0.0,A,NaN


In [4]:
df_fao_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 4209110 entries, 0 to 4209109
Data columns (total 14 columns):
 #   Column         Dtype  
---  ------         -----  
 0   Area_Code      int64  
 1   Area_Code_M49  str    
 2   Area           str    
 3   Item_Code      int64  
 4   Item_Code_CPC  str    
 5   Item           str    
 6   Element_Code   int64  
 7   Element        str    
 8   Year_Code      int64  
 9   Year           int64  
 10  Unit           str    
 11  Value          float64
 12  Flag           str    
 13  Note           str    
dtypes: float64(1), int64(5), str(8)
memory usage: 449.6 MB


In [5]:
# Remove the first character (the apostrophe) in the Area_Code_M49 and the Item_Code_CPC
df_fao_raw['Area_Code_M49'] = df_fao_raw['Area_Code_M49'].str.strip("'")
df_fao_raw['Item_Code_CPC'] = df_fao_raw['Item_Code_CPC'].str.strip("'")
df_fao_raw.head(3)

,Area_Code,Area_Code_M49,Area,Item_Code,Item_Code_CPC,Item,Element_Code,Element,Year_Code,Year,Unit,Value,Flag,Note
0,2,004,Afghanistan,221,01371,"Almonds, in shell",5312,Area harvested,1961,1961,ha,0.0,A,NaN
1,2,004,Afghanistan,221,01371,"Almonds, in shell",5312,Area harvested,1962,1962,ha,0.0,A,NaN
2,2,004,Afghanistan,221,01371,"Almonds, in shell",5312,Area harvested,1963,1963,ha,0.0,A,NaN


# 3.0 FAO Tea Data

In [6]:
# Show all rows in value_counts output
pd.set_option('display.max_rows', None)

## 3.1 Tea Items
All the entries with "tea" in its various forms:

In [7]:
tea_items = df_fao_raw['Item'].str.contains('tea',  na=False, case=False)
df_tea_items = df_fao_raw.loc[tea_items]

print(df_tea_items[['Item_Code','Item']].value_counts())

Item_Code  Item                                                                                                                            
667        Tea leaves                                                                                                                          13620
675        Green tea (not fermented), black tea (fermented) and partly fermented tea, in immediate packings of a content not exceeding 3 kg     1286
Name: count, dtype: int64


### 3.1.1 EDA of FAO Tea Data

#### 3.1.1.1 EDA of FAO Tea Data: Year and Item_Code Columns

Subset all the tea data based on `Item_Code` = 667 and also for `Year` 2000-2024 inclusive.

In [8]:
df_fao_tea = df_fao_raw[
    (df_fao_raw['Item_Code'] == 667) &
    ((df_fao_raw['Year'] >= 2000) &
    (df_fao_raw['Year'] <= 2024))
]

As the FAO bulk data file "Production_Crops_Livestock_E_All_Data_Normalized.csv" is 500+Mb in size, we will export the df_fao_tea as the cvs raw data.

In [20]:
df_fao_tea.to_csv(f'{raw_dir}/df_fao_tea.csv', index=False)
print(f"Exported: {raw_dir}/df_fao_tea.csv")

Exported: ../data/raw/df_fao_tea.csv


🔍 Now taking a look at the `Year` Column:
* noted there might be extraeous rows in years before 2017 and fewer rows in 2018-2021 and then 2022 onwards we have 2 more rows.
* this might even out after we look at the top 15 producting countries.

In [9]:
df_tea_items['Year'].value_counts()

Year
2022    382
2023    380
2021    372
2020    346
2019    296
2014    291
2017    291
2015    289
2016    288
2018    285
2010    282
2011    282
2012    282
2013    282
2006    224
1999    221
2000    221
2001    221
2002    221
2003    221
2004    221
2005    221
2007    221
2008    221
2009    221
1992    220
1993    220
1994    220
1995    220
1996    220
1997    220
1998    220
2024    217
1991    214
1990    213
1985    212
1986    212
1987    212
1988    212
1989    212
1976    208
1977    208
1978    208
1979    208
1980    208
1981    208
1982    208
1983    208
1984    208
1974    206
1975    206
1972    201
1973    201
1967    200
1968    200
1969    200
1970    200
1971    200
1961    199
1962    199
1963    199
1964    199
1965    199
1966    199
Name: count, dtype: int64

🔍 Now taking a look at the `Item_Code` Column:

In [ ]:
print("The List of Item Code in df_fao_tea:")
print(df_fao_tea['Item_Code'].value_counts())

The List of Item Code in df_fao_tea:
Item_Code
667    5492
Name: count, dtype: int64


#### 3.1.1.2 EDA of FAO Tea Data: Area

🔍 Now taking a look at the `Area` & `Area_Code_M49' Columns:
* We can see that the FAO bulk data has aggregation at "World" and other region areas level. 
* As we are look at the country granularity, we will need to explicitly exclude these area aggregation granularity when selecting the top 15 producing countries.

In [18]:
print("The List of Countries or Regions in df_fao_tea:")
print(df_fao_tea[['Area_Code_M49','Area']].value_counts())

The List of Countries or Regions in df_fao_tea:
Area_Code_M49  Area                                            
032            Argentina                                           75
031            Azerbaijan                                          75
050            Bangladesh                                          75
068            Bolivia (Plurinational State of)                    75
076            Brazil                                              75
108            Burundi                                             75
120            Cameroon                                            75
159            China                                               75
156            China, mainland                                     75
158            China, Taiwan Province of                           75
170            Colombia                                            75
180            Democratic Republic of the Congo                    75
218            Ecuador                          

#### 3.1.1.3 EDA of FAO Tea Data: Element

🔍 Now taking a look at the `Element`:
* The element values of are the measures for the respective country's output for the year. 
  * "Area harvested" (in hectares), 
  * "Production" (in tonnages) and 
  * "Yield" (in kilograms/hectare) etc 
* It is to be read in conjection with the `Unit` (unit of measurement) and the `Value` columns.
* In the Transformation section of for this tea data, we will do a check on the "Yield" column using the provided "Area harvested" and "Production".

In [11]:
print("The List of Elements in df_fao_tea:")
print(df_fao_tea[['Element', 'Unit']].value_counts())

The List of Elements in df_fao_tea:
Element         Unit 
Area harvested  ha       1840
Production      t        1840
Yield           kg/ha    1812
Name: count, dtype: int64


## 3.2 Transformation of FAO Tea Data and Export to CSV

### 3.2.1 Transformation of FAO Tea Data

The FAO bulk data file also include aggregrations rows in the data. They have been identified and as they are of different granularity to the country level data required by this project, we will need to remove them.

In [12]:
# create an exclude list for regions in the Area column 
exclude_areas = ['001'    # World
                , '002'    # Africa
                , '014'    # Eastern Africa
                , '017'    # Middle Africa
                , '018'    # Southern Africa
                , '011'    # Western Africa
                , '019'    # Americas
                , '013'    # Central America
                , '005'    # South America
                , '142'    # Asia
                , '030'    # Eastern Asia
                , '034'    # Southern Asia
                , '035'    # South-eastern Asia
                , '145'    # Western Asia
                , '150'    # Europe
                , '151'    # Eastern Europe
                , '039'    # Southern Europe
                , '009'    # Oceania
                , '054'    # Melanesia
                , '199'    # Least Developed Countries (LDCs)
                , '432'    # Land Locked Developing Countries (LLDCs)
                , '722'    # Small Island Developing States (SIDS)
                , '901'    # Low Income Food Deficit Countries (LIFDCs)
                , '902'    # Net Food Importing Developing Countries (NFIDCs)
                , '097'    # European Union (27)
                , '638'    # Réunion
                , '159'    # China (This code represents the aggregate of all Chinese territories for statistical purposes. It is the sum of Mainland China (156), Taiwan Province (214), Hong Kong SAR (96), and Macao SAR (128).)
]

In [13]:
# new dataset to only contain countries (by using the ~)
df_fao_tea_all_countries = df_fao_tea[~df_fao_tea['Area_Code_M49'].isin(exclude_areas)]

print("The List of Countries (exclude regions) in df_fao_tea:")
print(df_fao_tea_all_countries['Area'].value_counts())

The List of Countries (exclude regions) in df_fao_tea:
Area
Argentina                           75
Azerbaijan                          75
Bangladesh                          75
Bolivia (Plurinational State of)    75
Brazil                              75
Burundi                             75
Cameroon                            75
China, mainland                     75
China, Taiwan Province of           75
Colombia                            75
Democratic Republic of the Congo    75
Ecuador                             75
El Salvador                         75
Ethiopia                            75
Georgia                             75
Guatemala                           75
India                               75
Indonesia                           75
Iran (Islamic Republic of)          75
Japan                               75
Kenya                               75
Lao People's Democratic Republic    75
Madagascar                          75
Malawi                              75
Mala

As the MVP of this project is top 15 tea countries by `Production` value, we need to sum the countries' production over the 25 year period in the data set.

In [14]:
# Sum up production by country
df_tea_production_by_country = df_fao_tea_all_countries[df_fao_tea_all_countries['Element'] == 'Production'].groupby('Area')['Value'].sum().reset_index() 

# Rename the columns 
df_tea_production_by_country.columns = ["Country", "Sum_Production_25yrs"]

# Take a look at the data and set display for the Sum_Production_25yrs
pd.set_option('display.float_format', '{:,.0f}'.format)
df_tea_production_by_country.head(5)

,Country,Sum_Production_25yrs
0,Argentina,"8,556,455"
1,Azerbaijan,"20,877"
2,Bangladesh,"7,561,625"
3,Bolivia (Plurinational State of),"27,684"
4,Brazil,"308,408"


In [15]:
# sort the countries from high to low for column Sum_Production_25yrs
df_top_15_tea_country_only = df_tea_production_by_country.sort_values(
        "Sum_Production_25yrs", ascending=False
        ).head(15).reset_index(drop=True)

df_top_15_tea_country_only

,Country,Sum_Production_25yrs
0,"China, mainland","211,400,700"
1,India,"122,236,562"
2,Kenya,"45,303,100"
3,Sri Lanka,"37,877,670"
4,Türkiye,"30,054,368"
5,Viet Nam,"20,796,766"
6,Indonesia,"16,435,000"
7,Japan,"9,236,400"
8,Argentina,"8,556,455"
9,Bangladesh,"7,561,625"


In [16]:
# Get top 15 countries data only
df_fao_tea_top15_countries = df_fao_tea_all_countries[
        df_fao_tea_all_countries["Area"]
        .isin(df_top_15_tea_country_only["Country"].unique())]

df_fao_tea_top15_countries.head()


,Area_Code,Area_Code_M49,Area,Item_Code,Item_Code_CPC,Item,Element_Code,Element,Year_Code,Year,Unit,Value,Flag,Note
99510,9,032,Argentina,667,01620,Tea leaves,5312,Area harvested,2000,2000,ha,"38,620",A,NaN
99511,9,032,Argentina,667,01620,Tea leaves,5312,Area harvested,2001,2001,ha,"37,420",A,NaN
99512,9,032,Argentina,667,01620,Tea leaves,5312,Area harvested,2002,2002,ha,"36,400",A,NaN
99513,9,032,Argentina,667,01620,Tea leaves,5312,Area harvested,2003,2003,ha,"36,380",A,NaN
99514,9,032,Argentina,667,01620,Tea leaves,5312,Area harvested,2004,2004,ha,"36,670",A,NaN


In [17]:
# Double check the extraction only has 15 countries
df_fao_tea_top15_countries["Area"].value_counts()

Area
Argentina                      75
Bangladesh                     75
China, mainland                75
India                          75
Indonesia                      75
Iran (Islamic Republic of)     75
Japan                          75
Kenya                          75
Malawi                         75
Rwanda                         75
Sri Lanka                      75
Türkiye                        75
Uganda                         75
United Republic of Tanzania    75
Viet Nam                       75
Name: count, dtype: int64

In [26]:
# Check that each of the top 15 countries has 3 of the `Element` measures
elements_by_country = df_fao_tea_top15_countries.groupby("Area")["Element"].unique().reset_index()

print("Elements available for each of the top 15 countries")
print("===================================================")
for index, row in elements_by_country.iterrows():
    print(f"For The Country of {row['Area']}:")
    for element in row["Element"]:
        print(f" * {element}")
    print("-------------------------")    

Elements available for each of the top 15 countries
For The Country of Argentina:
 * Area harvested
 * Yield
 * Production
-------------------------
For The Country of Bangladesh:
 * Area harvested
 * Yield
 * Production
-------------------------
For The Country of China, mainland:
 * Area harvested
 * Yield
 * Production
-------------------------
For The Country of India:
 * Area harvested
 * Yield
 * Production
-------------------------
For The Country of Indonesia:
 * Area harvested
 * Yield
 * Production
-------------------------
For The Country of Iran (Islamic Republic of):
 * Area harvested
 * Yield
 * Production
-------------------------
For The Country of Japan:
 * Area harvested
 * Yield
 * Production
-------------------------
For The Country of Kenya:
 * Area harvested
 * Yield
 * Production
-------------------------
For The Country of Malawi:
 * Area harvested
 * Yield
 * Production
-------------------------
For The Country of Rwanda:
 * Area harvested
 * Yield
 * Productio

### 3.2.2 Convert the Country Column into ISO3 Standard

In [18]:
import country_converter as coco

cc = coco.CountryConverter()

df_fao_tea_top15_countries["Country_ISO3"] = cc.pandas_convert(
    series=df_fao_tea_top15_countries['Area'],
    to='ISO3'
)

df_fao_tea_top15_countries["Country_ISO3"].value_counts()

Country_ISO3
ARG    75
BGD    75
CHN    75
IND    75
IDN    75
IRN    75
JPN    75
KEN    75
MWI    75
RWA    75
LKA    75
TUR    75
UGA    75
TZA    75
VNM    75
Name: count, dtype: int64

### 3.2.3 Export FAO Tea Data to CSV for Analysis

In [19]:
df_fao_tea_top15_countries.to_csv(f'{processed_dir}/df_fao_tea_top15_countries.csv', index=False)
print(f"Exported: {processed_dir}/df_fao_tea_top15_countries.csv")

Exported: ../data/processed/df_fao_tea_top15_countries.csv
